# Version 3 of the Titanic Competition

We use the Pipeline with GridSearch (to get best params) and StratifiedKFold (to evaluation the models)

## Import Libraries and Data

In [20]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.compose import ColumnTransformer 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold,cross_validate
from sklearn.base import BaseEstimator,TransformerMixin,clone
# Models we will test
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [2]:
train_data = pd.read_csv('./Data/train.csv')
test_data  = pd.read_csv('./Data/test.csv')
print(train_data.shape,test_data.shape)

(891, 12) (418, 11)


## Explore Data Analysis (EDA)

## Make the Pipeline

We group the similar feature to perform the preprocessing in each of them

In [3]:
num_features = ['Age','Fare']
cat_features = ['Embarked','titles','Sex']
pass_through = ['Has_Family','Pclass']
Selected_Feature = ["Pclass","Sex","Age","Fare","Embarked","titles","Cabin_Count","Has_Family","Family_Size"]

In [4]:
'''
In Pre_Stage :
1. Get the Titles from the name feature
2. Imputation of Age by the Titles of the names
3.Feature Engineering by Selecting and make a new features
'''
class Pre_Stage(BaseEstimator,TransformerMixin):
    def __init__(self,selected_features):
        self.selected_features = selected_features
    def _get_title(self,name_feature):
        common_titles = ['Mr','Miss','Mrs','Master']
        titles = name_feature.apply(lambda x : x.split(',')[1].split('.')[0].strip())
        titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        return titles.where(titles.isin(common_titles),'Rare')
    def fit(self,X,y=None):
        df = X.copy()
        titles = self._get_title(df.Name)
        self.title_age_median = df.groupby(titles).Age.median().to_dict()
        self.global_age_median = df.Age.median()
        return self
    def transform(self,X):
        df = X.copy()
        df['titles'] = self._get_title(df.Name)
        df.Age = df.Age.fillna(df.titles.map(self.title_age_median)).fillna(self.global_age_median)
        # Feature Engineering (create new features)
        sibsp = df.SibSp.fillna(0)
        parch = df.Parch.fillna(0)
        df['Family_Size'] = sibsp + parch + 1
        df['Has_Family']  = (df['Family_Size'] > 1).astype(int)
        df['Cabin_Count'] = df.Cabin.apply(lambda x : len(x.split(' ')) if pd.notna(x) else 0)
        # Feature selection (select a few features thta matter)
        return df[self.selected_features]
        
        
        
    

In [5]:
num_Pipeline = Pipeline(steps=[('SimpleImputer',SimpleImputer(strategy='median')),
                               ('scaler',StandardScaler())])
cat_Pipelien = Pipeline(steps=[('SimpleImputer',SimpleImputer(strategy='most_frequent')),
                               ('encoding',OneHotEncoder(handle_unknown='ignore',sparse_output=False,drop="if_binary"))])

In [6]:
Preprocessor = ColumnTransformer(transformers=[('Num',num_Pipeline,num_features),
                                               ('Pclass',SimpleImputer(strategy='most_frequent'),['Pclass']),
                                               ('Cat',cat_Pipelien,cat_features),
                                               ('pass','passthrough',pass_through)],verbose_feature_names_out=True)

In [7]:
final_pipeline = Pipeline(steps=[('preStage',Pre_Stage(Selected_Feature)),
                                 ('preprocessing',Preprocessor),
                                 ('model',RandomForestClassifier())])

In [8]:
from sklearn.model_selection import train_test_split
X = train_data.drop(['Survived'],axis=1)
y = train_data.Survived
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y)

We just fit the pipeline to see the tree of the Pipline

In [9]:
final_pipeline.fit(X_train,y_train)

,steps,"[('preStage', ...), ('preprocessing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,selected_features,"['Pclass', 'Sex', ...]"
,transformers,"[('Num', ...), ('Pclass', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [10]:
SCV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = cross_validate(
    final_pipeline,
    X_train,
    y_train,
    cv=SCV,
    scoring=["accuracy", "f1"],
    return_train_score=True,
)

print(f'Train Acc :{results['train_accuracy'].mean()} , Test Acc : {results['test_accuracy'].mean()}')
print(f'Train f1 :{results['train_f1'].mean()} , Test f1 : {results['test_f1'].mean()}')

Train Acc :0.9824431905774981 , Test Acc : 0.8117502216093765
Train f1 :0.977052908546144 , Test f1 : 0.7447922922922923


In [11]:
final_pipeline.score(X_test,y_test)

0.8100558659217877

* We have a Overfitting that the training accuracy is **~15%** higher that testing accuracy 
* We will make **Grid Search** to See the best parameters that make higher accuracy in Testing and prevent overfitting

In [12]:
param_grid = [
    {
        "model": [RandomForestClassifier(random_state=42)],
        "model__n_estimators": [100, 200],
        "model__max_depth": [4, 6, 8],
        "model__min_samples_leaf": [2, 4],
    },
    {
        "model": [GradientBoostingClassifier(random_state=42)],
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 4],
    },
    {
        "model": [LogisticRegression(max_iter=1000, random_state=42)],
        "model__C": [0.1, 1.0, 10.0],
        "model__penalty": ["l2"],
    },
    {
        "model": [SVC(random_state=42)],
        "model__C": [0.5, 1.0, 5.0],
        "model__kernel": ["rbf", "linear"],
    },
]
grid_search = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid,
    cv=SCV,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"[{'model': [RandomForestC...ndom_state=42)], 'model__max_depth': [4, 6, ...], 'model__min_samples_leaf': [2, 4], 'model__n_estimators': [100, 200]}, {'model': [GradientBoost...ndom_state=42)], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 4], 'model__n_estimators': [100, 200]}, ...]"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,selected_features,"['Pclass', 'Sex', ...]"


In [13]:
grid_df = pd.DataFrame(grid_search.cv_results_)
grid_df['model_name'] = grid_df.param_model.apply(lambda x : type(x).__name__)
best_param_df = grid_df[['model_name','mean_train_score','mean_test_score']].sort_values(by=['mean_test_score'],ascending=False).reset_index()[:5]

In [14]:
best_param_df

,index,model_name,mean_train_score,mean_test_score
0,10,RandomForestClassifier,0.870785,0.830060
1,4,RandomForestClassifier,0.868329,0.830001
2,14,GradientBoostingClassifier,0.916082,0.828632
3,9,RandomForestClassifier,0.892201,0.828612
4,12,GradientBoostingClassifier,0.886235,0.827233


In [15]:
Score = grid_search.best_score_
print(f'The Score of best model {best_param_df.model_name[0]}: {Score:.6f}')
print(f'The Score on the test data : {grid_search.best_estimator_.score(X_test,y_test):.6f}')

The Score of best model RandomForestClassifier: 0.830060
The Score on the test data : 0.843575


In [16]:
y_preds = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test,y_preds))

              precision    recall  f1-score   support

           0       0.85      0.91      0.88       110
           1       0.84      0.74      0.78        69

    accuracy                           0.84       179
   macro avg       0.84      0.82      0.83       179
weighted avg       0.84      0.84      0.84       179



In [22]:
PassengerIds = test_data.PassengerId
final_model = clone(grid_search.best_estimator_)
final_model.fit(X,y)
test_preds = final_model.predict(test_data)
test_preds

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0,

In [28]:
submission = pd.DataFrame({'PassengerId':PassengerIds,
                           'Survived':test_preds.astype(int)})
submission.to_csv('Submission_V3.csv',index=False)
print(submission.shape)
print(submission.head(5))
print(submission.Survived.value_counts())

(418, 2)
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Survived
0    266
1    152
Name: count, dtype: int64
